# Hackathon Judge SFT — Kaggle Smoke Test
Small model, low rank, tiny dataset. Runs end-to-end in ~10 min on T4.

In [ ]:
# ── params ────────────────────────────────────────────────
MODEL      = "unsloth/Qwen3-0.6B"
HACKATHON  = "madhacks"   # smallest config (~460 trainable rows)
N_TRAIN    = 40           # pairs to train on  (80 rows with ab+ba)
N_EVAL     = 10           # pairs to evaluate
LORA_R     = 8
EPOCHS     = 1
OUTPUT_DIR = "/kaggle/working/lora_test"
# ──────────────────────────────────────────────────────────

In [ ]:
%%capture
!git clone https://github.com/woopxwoop/cs639-hackathon-judge-ft.git /kaggle/working/ft
!pip install "unsloth[kaggle-new]" trl "datasets>=4.8.5" pyarrow peft
!pip install -e /kaggle/working/ft --no-deps

In [ ]:
import random
from hackathon_judge_ft.data import load_frontier, split

ds = load_frontier(HACKATHON)
pairs = list(set(ds["pair_id"]))
random.seed(42)
random.shuffle(pairs)
train_pairs = set(pairs[:N_TRAIN])
eval_pairs  = set(pairs[N_TRAIN:N_TRAIN + N_EVAL])

trainable = ds.filter(lambda r: r["verdict"] in ("A", "B"))
train_ds = trainable.filter(lambda r: r["pair_id"] in train_pairs)
eval_ds  = trainable.filter(lambda r: r["pair_id"] in eval_pairs)
print(f"train={len(train_ds)}  eval={len(eval_ds)}")

In [ ]:
from hackathon_judge_ft.train import run as train_run

train_run(train_ds, model_name=MODEL, output_dir=OUTPUT_DIR, epochs=EPOCHS, r=LORA_R)
print(f"saved → {OUTPUT_DIR}")

In [ ]:
import torch
from unsloth import FastModel
from peft import PeftModel
from hackathon_judge_ft.evaluate import parse_verdict

base, tokenizer = FastModel.from_pretrained(MODEL, max_seq_length=8192, load_in_4bit=True)
eval_model = PeftModel.from_pretrained(base, OUTPUT_DIR)
eval_model.eval()

def infer(messages):
    prompt = tokenizer.apply_chat_template(
        [m for m in messages if m["role"] != "assistant"],
        tokenize=False, add_generation_prompt=True, enable_thinking=True,
    )
    inputs = tokenizer(prompt, return_tensors="pt").to(eval_model.device)
    with torch.no_grad():
        out = eval_model.generate(**inputs, max_new_tokens=2048, temperature=0.7, top_p=0.9, do_sample=True)
    return tokenizer.decode(out[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)

In [ ]:
results = []
for ex in eval_ds:
    response = infer(ex["messages"])
    predicted = parse_verdict(response)
    match = predicted == ex["verdict"]
    results.append({"pair_id": ex["pair_id"], "position": ex["position"],
                    "predicted": predicted, "frontier": ex["verdict"], "match": match})
    print(f"{ex['pair_id'][:8]} pos={ex['position']}  predicted={predicted}  frontier={ex['verdict']}  {'✓' if match else '✗'}")

agreement = sum(r["match"] for r in results) / len(results)
pair_map = {}
for r in results:
    pair_map.setdefault(r["pair_id"], {})[r["position"]] = r["predicted"]
consistent = sum(
    1 for v in pair_map.values()
    if (v.get("ab") == "A" and v.get("ba") == "B") or (v.get("ab") == "B" and v.get("ba") == "A")
)
n_pairs = sum(1 for v in pair_map.values() if "ab" in v and "ba" in v)

print(f"\nfrontier agreement:   {agreement*100:.1f}% ({len(results)} examples)")
print(f"position consistency: {consistent/n_pairs*100:.0f}% ({n_pairs} pairs)")